In [1]:
"""Create candidate nematode crops from the genus-labelled microscope images.

These are *pseudo* bounding boxes: visually review the images in `--preview-dir`
before using the crops as training data.
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import cv2
import numpy as np


def organism_boxes(image: np.ndarray, min_area: int = 1_500) -> list[tuple[int, int, int, int]]:
    """Return contour boxes likely to contain a darker elongated organism."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Local threshold tolerates uneven microscope illumination. In this data,
    # nematodes are typically darker than their bright-field background.
    mask = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 51, 5
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    height, width = gray.shape
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        aspect = max(w, h) / max(1, min(w, h))
        # Reject specks, border/background regions, and near-square debris.
        if min_area <= area <= 0.80 * width * height and aspect >= 1.4:
            boxes.append((x, y, w, h))
    return sorted(boxes, key=lambda b: b[2] * b[3], reverse=True)


def padded_crop(image: np.ndarray, box: tuple[int, int, int, int], padding: float = 0.12) -> np.ndarray:
    x, y, w, h = box
    pad = int(max(w, h) * padding)
    y0, y1 = max(0, y - pad), min(image.shape[0], y + h + pad)
    x0, x1 = max(0, x - pad), min(image.shape[1], x + w + pad)
    return image[y0:y1, x0:x1]


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, default=Path("dataset"))
    parser.add_argument("--crops-dir", type=Path, default=Path("artifacts/crops"))
    parser.add_argument("--preview-dir", type=Path, default=Path("artifacts/previews"))
    parser.add_argument("--csv", type=Path, default=Path("artifacts/pseudo_boxes.csv"))
    parser.add_argument("--min-area", type=int, default=1500)
    parser.add_argument("--max-boxes", type=int, default=1,
                        help="Use 1 when each source image contains one nematode.")
    args = parser.parse_args()
    args.csv.parent.mkdir(parents=True, exist_ok=True)

    with args.csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "genus", "magnification", "x", "y", "width", "height", "crop", "reviewed"])
        writer.writeheader()
        for path in sorted(args.input.glob("Genus */*.jpg")):
            image = cv2.imread(str(path))
            if image is None:
                print(f"Skipping unreadable file: {path}")
                continue
            # Folder names are the ground-truth genus labels in this dataset.
            genus = path.parent.name.removeprefix("Genus ")
            boxes = organism_boxes(image, args.min_area)[:args.max_boxes]
            preview = image.copy()
            for index, (x, y, w, h) in enumerate(boxes):
                crop_path = args.crops_dir / genus / f"{path.stem}_{index}.jpg"
                crop_path.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(crop_path), padded_crop(image, (x, y, w, h)))
                cv2.rectangle(preview, (x, y), (x + w, y + h), (0, 255, 0), 3)
                writer.writerow({"image": str(path), "genus": genus, "magnification": "",
                                 "x": x, "y": y, "width": w, "height": h,
                                 "crop": str(crop_path), "reviewed": "no"})
            preview_path = args.preview_dir / genus / path.name
            preview_path.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(preview_path), preview)


In [2]:
"""Fine-tune an ImageNet-pretrained classifier on reviewed nematode crops."""
from __future__ import annotations

import argparse
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, default=Path("artifacts/crops"))
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--output", type=Path, default=Path("artifacts/nematode_resnet18.pt"))
    args = parser.parse_args()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    dataset = datasets.ImageFolder(args.data, transform=transform)
    if len(dataset.classes) < 2:
        raise ValueError("Need crops from at least two genus folders.")
    n_val = max(1, round(len(dataset) * 0.2))
    train_set, val_set = random_split(dataset, [len(dataset) - n_val, n_val], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=2)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(args.epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(x.to(device)), y.to(device))
            loss.backward(); optimizer.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in val_loader:
                predicted = model(x.to(device)).argmax(1).cpu()
                correct += (predicted == y).sum().item(); total += y.numel()
        print(f"epoch {epoch + 1}/{args.epochs}: validation accuracy {correct / total:.1%}")

    args.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "classes": dataset.classes}, args.output)


In [3]:
# Run this cell after the OpenCV helper cell above.
# It saves only full-size images with boxes: no crops and no class folders.
from pathlib import Path
import csv
import cv2

INPUT_DIR = Path('dataset')
OUTPUT_DIR = Path('artifacts/boxed_images')  # one flat folder
BOX_CSV = Path('artifacts/pseudo_boxes.csv')
MIN_AREA = 1500
MAX_BOXES = 1  # increase only when source images can contain multiple nematodes

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BOX_CSV.parent.mkdir(parents=True, exist_ok=True)
saved = 0
no_box = 0

with BOX_CSV.open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        'image', 'genus', 'magnification', 'x', 'y', 'width', 'height',
        'boxed_image', 'reviewed'
    ])
    writer.writeheader()

    for path in sorted(INPUT_DIR.glob('Genus */*.jpg')):
        image = cv2.imread(str(path))
        if image is None:
            print(f'Skipping unreadable image: {path}')
            continue

        genus = path.parent.name.removeprefix('Genus ')
        boxes = organism_boxes(image, MIN_AREA)[:MAX_BOXES]
        annotated = image.copy()

        if not boxes:
            no_box += 1
            cv2.putText(annotated, 'NO BOX FOUND', (30, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

        for x, y, w, h in boxes:
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 3)

        # Genus prefix prevents duplicate names while keeping artifacts flat.
        output_path = OUTPUT_DIR / f'{genus}__{path.name}'
        cv2.imwrite(str(output_path), annotated)
        saved += 1

        for x, y, w, h in boxes:
            writer.writerow({
                'image': str(path), 'genus': genus, 'magnification': '',
                'x': x, 'y': y, 'width': w, 'height': h,
                'boxed_image': str(output_path), 'reviewed': 'no'
            })

print(f'Saved {saved} annotated full images to {OUTPUT_DIR}')
print(f'Images with no candidate box: {no_box}')
print(f'Box table: {BOX_CSV}')


Saved 1016 annotated full images to artifacts/boxed_images
Images with no candidate box: 142
Box table: artifacts/pseudo_boxes.csv
